# 18 - Fundraising Agent
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/18-fundraising-agent/fundraising_agent_workbook.ipynb)

Generate investor-ready fundraising materials from one company brief -- tailored separately for VCs, PE firms, and family offices.

**Harness focus:** Audience-targeted generation -- same data, different typed output per persona

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: Same data, different audience

The same company facts land differently depending on who is reading them:

| Investor | Primary concern | Language |
|----------|-----------------|----------|
| VC | Growth velocity + TAM | ARR, NRR, burn multiple |
| PE | EBITDA + exit multiple | EBITDA margin, EV/EBITDA, leverage |
| Family office | Capital preservation | Downside, dividend, governance |

The pattern: three separate LLM calls, each with a persona-specific system prompt, all returning the same `FundraisingMaterials` schema. The content is shaped entirely differently for each audience.

## Part 2: Schema -- FundraisingMaterials and FundraisingPackage

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel

class FundraisingMaterials(BaseModel):
    target_persona: Literal["vc", "pe", "family_office"]
    investor_thesis: str
    headline_metrics: List[str]
    narrative_angle: str
    key_asks: List[str]
    objection_responses: List[str]
    suggested_materials: List[str]

class FundraisingPackage(BaseModel):
    company: Optional[str] = None
    round_type: str
    vc_materials: FundraisingMaterials
    pe_materials: FundraisingMaterials
    family_office_materials: FundraisingMaterials
    universal_value_props: List[str]

print("Schema defined.")

## Part 3: Three persona-specific system prompts

Each prompt frames the same company data through a different investor lens.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

PERSONA_PROMPTS = {
    "vc": (
        "You are drafting fundraising materials for a VENTURE CAPITAL investor. "
        "VCs want TAM, growth velocity, and founder-market fit. "
        "Speak in ARR, NRR, burn multiple, and CAC payback."
    ),
    "pe": (
        "You are drafting fundraising materials for a PRIVATE EQUITY investor. "
        "PE firms want EBITDA and a clear exit multiple. Avoid hype. "
        "Frame the story around margin expansion and downside protection."
    ),
    "family_office": (
        "You are drafting fundraising materials for a FAMILY OFFICE investor. "
        "They want capital preservation and stable compounding. "
        "Frame the story around resilience and governance quality."
    ),
}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Persona prompts ready.")

## Part 4: Run all three persona calls

In [ ]:
BRIEF = """
COMPANY: Volt Energy Technologies Ltd
Business: B2B energy management SaaS  |  Employees: 38
ARR: GBP 8.2m (+67% YoY)  |  Gross margin: 74%  |  NRR: 118%
Burn: GBP 380k/month  |  Runway: 14 months
210 customers; largest = 31% of ARR
TAM: GBP 2.8bn UK commercial energy management
Team: CEO (ex-energy 12yr), CTO (ex-DeepMind), CFO (ex-Big 4)
Seeking: Series B GBP 15m
"""

def draft_for_persona(persona):
    system = SystemMessage(PERSONA_PROMPTS[persona])
    drafter = system | llm.with_structured_output(FundraisingMaterials)
    return drafter.invoke(
        HumanMessage(
            content=f"Company brief:\n\n{BRIEF}\n\nDraft materials for target_persona='{persona}'"
        )
    )

vc = draft_for_persona("vc")
pe = draft_for_persona("pe")
fo = draft_for_persona("family_office")

package = FundraisingPackage(
    round_type="Series B",
    vc_materials=vc,
    pe_materials=pe,
    family_office_materials=fo,
    universal_value_props=vc.headline_metrics[:3],
)
print("Package assembled.")

## Part 5: Compare outputs side by side

In [ ]:
def show_persona(label, mat):
    print(f"\n{'=' * 50}")
    print(f"FOR: {label}")
    print(f"{'=' * 50}")
    print(f"\nThesis: {mat.investor_thesis}")
    print("\nHeadline metrics:")
    for m in mat.headline_metrics:
        print(f"  - {m}")
    print(f"\nNarrative: {mat.narrative_angle}")
    if mat.objection_responses:
        print(f"\nTop objection: {mat.objection_responses[0]}")

if package.universal_value_props:
    print("UNIVERSAL VALUE PROPS:")
    for v in package.universal_value_props:
        print(f"  + {v}")

show_persona("VENTURE CAPITAL", package.vc_materials)
show_persona("PRIVATE EQUITY", package.pe_materials)
show_persona("FAMILY OFFICE", package.family_office_materials)

## Exercise: Add a strategic investor persona

A strategic investor (corporate VC from a large energy company) cares about:
- Technology access and IP licensing potential
- Partnership and distribution opportunities

Add `strategic_materials: FundraisingMaterials` to `FundraisingPackage` and write a persona prompt for it.

**Starter:** Add the field below.

In [ ]:
# Starter: extend FundraisingPackage with strategic_materials
class FundraisingPackageV2(BaseModel):
    company: Optional[str] = None
    round_type: str
    vc_materials: FundraisingMaterials
    pe_materials: FundraisingMaterials
    family_office_materials: FundraisingMaterials
    strategic_materials: FundraisingMaterials  # NEW
    universal_value_props: List[str]

# TODO: write a persona prompt for 'strategic' (corporate VC, energy company)
# TODO: call draft_for_persona with the new prompt
# TODO: assemble FundraisingPackageV2 and call show_persona

In [ ]:
# Answer key
STRATEGIC_PROMPT = (
    "You are drafting fundraising materials for a STRATEGIC INVESTOR "
    "(corporate VC from a large energy company). They want technology access, "
    "IP licensing potential, and distribution partnerships. "
    "Frame the story around complementary assets and strategic synergies."
)

system_strategic = SystemMessage(STRATEGIC_PROMPT)
drafter_strategic = system_strategic | llm.with_structured_output(FundraisingMaterials)
strategic = drafter_strategic.invoke(
    HumanMessage(
        content=f"Company brief:\n\n{BRIEF}\n\nDraft materials for target_persona='strategic'"
    )
)

package_v2 = FundraisingPackageV2(
    round_type="Series B",
    vc_materials=vc,
    pe_materials=pe,
    family_office_materials=fo,
    strategic_materials=strategic,
    universal_value_props=vc.headline_metrics[:3],
)
show_persona("STRATEGIC INVESTOR", package_v2.strategic_materials)

## What You Built

- Audience-targeted generation: one company brief, three typed outputs
- Three persona-specific system prompts shaping the same schema differently
- `FundraisingPackage` assembling all outputs into one typed object
- A comparison showing how the same data lands differently per audience

**Congratulations** -- you have completed all 18 agent use cases in this repo!

Each example has isolated one harness attribute. Combine them to build production-grade agentic systems.